In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Model Training for DDoS Detection\n",
    "\n",
    "This notebook trains and evaluates machine learning models for DDoS attack detection using the engineered features."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split, cross_val_score\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.svm import SVC\n",
    "from xgboost import XGBClassifier\n",
    "from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve\n",
    "import joblib\n",
    "\n",
    "# Load engineered features\n",
    "# df = pd.read_parquet('data/processed/engineered_features.parquet')\n",
    "\n",
    "# For demonstration, we'll generate synthetic data with features similar to previous notebook\n",
    "np.random.seed(42)\n",
    "n_samples = 5000\n",
    "X = pd.DataFrame({\n",
    "    'bytes_sum': np.random.exponential(5000, n_samples),\n",
    "    'bytes_mean': np.random.exponential(200, n_samples),\n",
    "    'bytes_std': np.random.exponential(100, n_samples),\n",
    "    'packets_sum': np.random.poisson(100, n_samples),\n",
    "    'packets_mean': np.random.poisson(10, n_samples),\n",
    "    'packets_std': np.random.poisson(5, n_samples),\n",
    "    'duration_mean': np.random.exponential(0.5, n_samples),\n",
    "    'duration_std': np.random.exponential(0.2, n_samples),\n",
    "    'src_ip_entropy': np.random.uniform(0, 5, n_samples),\n",
    "    'dst_ip_entropy': np.random.uniform(0, 5, n_samples),\n",
    "    'src_port_entropy': np.random.uniform(0, 4, n_samples),\n",
    "    'dst_port_entropy': np.random.uniform(0, 4, n_samples),\n",
    "    'protocol_entropy': np.random.uniform(0, 2, n_samples),\n",
    "    'packet_size_mean': np.random.exponential(500, n_samples),\n",
    "    'packet_size_std': np.random.exponential(100, n_samples),\n",
    "    'small_packet_mean': np.random.beta(0.2, 0.8, n_samples),\n",
    "    'syn_ratio': np.random.beta(0.1, 0.9, n_samples),\n",
    "    'rst_ratio': np.random.beta(0.05, 0.95, n_samples),\n",
    "    'fin_ratio': np.random.beta(0.05, 0.95, n_samples),\n",
    "    'ack_ratio': np.random.beta(0.8, 0.2, n_samples),\n",
    "})\n",
    "\n",
    "# Create target (attack: 1, benign: 0)\n",
    "# For synthetic data, we'll simulate based on certain conditions\n",
    "attack_prob = (X['src_ip_entropy'] < 2) & (X['packets_sum'] > 150) & (X['syn_ratio'] > 0.5)\n",
    "y = (attack_prob | (X['dst_port_entropy'] < 1.5) | (X['small_packet_mean'] > 0.7)).astype(int)\n",
    "\n",
    "print(f\"Dataset: {X.shape[0]} samples, {X.shape[1]} features\")\n",
    "print(f\"Attack ratio: {y.mean():.3f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Split data\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n",
    "print(f\"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train Random Forest\n",
    "rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)\n",
    "rf.fit(X_train, y_train)\n",
    "y_pred_rf = rf.predict(X_test)\n",
    "y_proba_rf = rf.predict_proba(X_test)[:, 1]\n",
    "\n",
    "print(\"Random Forest Performance:\")\n",
    "print(classification_report(y_test, y_pred_rf))\n",
    "print(f\"AUC-ROC: {roc_auc_score(y_test, y_proba_rf):.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train XGBoost\n",
    "xgb = XGBClassifier(n_estimators=200, max_depth=10, learning_rate=0.1, random_state=42)\n",
    "xgb.fit(X_train, y_train)\n",
    "y_pred_xgb = xgb.predict(X_test)\n",
    "y_proba_xgb = xgb.predict_proba(X_test)[:, 1]\n",
    "\n",
    "print(\"XGBoost Performance:\")\n",
    "print(classification_report(y_test, y_pred_xgb))\n",
    "print(f\"AUC-ROC: {roc_auc_score(y_test, y_proba_xgb):.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare ROC curves\n",
    "fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)\n",
    "fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)\n",
    "\n",
    "plt.figure(figsize=(8,6))\n",
    "plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, y_proba_rf):.3f})')\n",
    "plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {roc_auc_score(y_test, y_proba_xgb):.3f})')\n",
    "plt.plot([0, 1], [0, 1], 'k--')\n",
    "plt.xlabel('False Positive Rate')\n",
    "plt.ylabel('True Positive Rate')\n",
    "plt.title('ROC Curve')\n",
    "plt.legend()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature importance from Random Forest\n",
    "importances = pd.Series(rf.feature_importances_, index=X.columns)\n",
    "importances.sort_values().tail(20).plot(kind='barh', figsize=(10,8))\n",
    "plt.title('Top 20 Feature Importances (Random Forest)')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cross-validation\n",
    "cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='f1')\n",
    "cv_scores_xgb = cross_val_score(xgb, X_train, y_train, cv=5, scoring='f1')\n",
    "print(f\"Random Forest CV F1: {cv_scores_rf.mean():.4f} +/- {cv_scores_rf.std():.4f}\")\n",
    "print(f\"XGBoost CV F1: {cv_scores_xgb.mean():.4f} +/- {cv_scores_xgb.std():.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save the best model (Random Forest for this demo)\n",
    "# joblib.dump(rf, 'models/rf_model.joblib')\n",
    "# joblib.dump(X_train.columns.tolist(), 'models/feature_names.joblib')\n",
    "print(\"Model saved\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}